# Session Signatures Debug
Shows the tower signature and assigned problem number for every session, for every subject.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.behavior_import.import_data import import_data
from src.behavior_import.extract_trials import (
    extract_trials_grouped_by_problem,
    choice_towers_signature,
    permutation_invariant_signature,
)
from fix_grid_maze_cohort_02_problems import fix_grid_maze_cohort_02_problems

folder_name = "3x3_maze_blocked_reward_bandit"
cohort = "cohort-02"
root = f"/Volumes/behrens/meg/{folder_name}/{cohort}/rawdata/"

subjects_data = import_data(root)
subjects_trials_by_problem = extract_trials_grouped_by_problem(subjects_data)
subjects_trials_by_problem = fix_grid_maze_cohort_02_problems(subjects_trials_by_problem)

[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-27_date-20260124/behav/._MY_04_R-2026-01-24-124422.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-55_date-20260209/behav/._MY_04_R-2026-02-09-103918.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-28_date-20260124/behav/._MY_04_R-2026-01-24-165301.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volumes/behrens/meg/3x3_maze_blocked_reward_bandit/cohort-02/rawdata/sub-03_id-MY_04_R/ses-12_date-20260116/behav/._MY_04_R-2026-01-16-143640.tsv: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte.
[WARN] Failed to read /Volum

In [2]:
import pandas as pd
pd.set_option("display.max_rows", None)

def session_num(ses_key):
    return int(ses_key.split("ses-")[1].split("_")[0])

rows = []
for prob_num, prob_subjects in sorted(subjects_trials_by_problem.items()):
    for subj, sessions in sorted(prob_subjects.items()):
        for ses_key, sess in sorted(sessions.items(), key=lambda x: session_num(x[0])):
            raw = choice_towers_signature(sess)
            sig = permutation_invariant_signature(raw)
            rows.append({
                "subject": subj,
                "session": ses_key,
                "ses_num": session_num(ses_key),
                "problem": prob_num,
                "signature": sig,
            })

df = pd.DataFrame(rows).sort_values(["subject", "ses_num"]).reset_index(drop=True)
pd.set_option("display.max_rows", None)
df

,subject,session,ses_num,problem,signature
0,MY_04_L,ses-1_date-20260111,1,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))"
1,MY_04_L,ses-2_date-20260111,2,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))"
2,MY_04_L,ses-3_date-20260112,3,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))"
3,MY_04_L,ses-4_date-20260112,4,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))"
4,MY_04_L,ses-5_date-20260113,5,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))"
5,MY_04_L,ses-6_date-20260113,6,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))"
6,MY_04_L,ses-7_date-20260114,7,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))"
7,MY_04_L,ses-8_date-20260114,8,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))"
8,MY_04_L,ses-9_date-20260115,9,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))"
9,MY_04_L,ses-10_date-20260115,10,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))"


## Per-subject view
Shows each subject's sessions in chronological order with their assigned problem number and signature.

In [3]:
pd.set_option("display.max_rows", None)

for subj, grp in df.groupby("subject"):
    grp = grp.sort_values("ses_num").reset_index(drop=True)
    grp["prob_changed"] = grp["problem"] != grp["problem"].shift()
    grp["is_revisit"] = grp.duplicated(subset=["subject", "signature"], keep="first") & grp["signature"].notna()

    print(f"\n{'='*60}")
    print(f"  {subj}")
    print(f"{'='*60}")
    display(grp[["session", "ses_num", "problem", "signature", "prob_changed", "is_revisit"]])


  MY_04_L


,session,ses_num,problem,signature,prob_changed,is_revisit
0,ses-1_date-20260111,1,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",True,False
1,ses-2_date-20260111,2,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
2,ses-3_date-20260112,3,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
3,ses-4_date-20260112,4,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
4,ses-5_date-20260113,5,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
5,ses-6_date-20260113,6,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
6,ses-7_date-20260114,7,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
7,ses-8_date-20260114,8,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
8,ses-9_date-20260115,9,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
9,ses-10_date-20260115,10,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True



  MY_04_N


,session,ses_num,problem,signature,prob_changed,is_revisit
0,ses-1_date-20260111,1,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",True,False
1,ses-2_date-20260111,2,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
2,ses-3_date-20260112,3,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
3,ses-4_date-20260112,4,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
4,ses-5_date-20260113,5,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
5,ses-6_date-20260113,6,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
6,ses-7_date-20260114,7,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
7,ses-8_date-20260114,8,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
8,ses-9_date-20260115,9,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
9,ses-10_date-20260115,10,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True



  MY_04_R


...

,session,ses_num,problem,signature,prob_changed,is_revisit
0,ses-1_date-20260111,1,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",True,False
1,ses-2_date-20260111,2,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
2,ses-3_date-20260112,3,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
3,ses-4_date-20260112,4,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
4,ses-5_date-20260113,5,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
5,ses-6_date-20260113,6,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
6,ses-7_date-20260114,7,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
7,ses-8_date-20260114,8,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
8,ses-9_date-20260115,9,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
9,ses-10_date-20260115,10,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True



  MY_05_N


,session,ses_num,problem,signature,prob_changed,is_revisit
0,ses-2_date-20260111,2,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",True,False
1,ses-3_date-20260112,3,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
2,ses-4_date-20260112,4,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
3,ses-5_date-20260113,5,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
4,ses-6_date-20260113,6,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
5,ses-7_date-20260114,7,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
6,ses-8_date-20260114,8,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
7,ses-9_date-20260115,9,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
8,ses-10_date-20260115,10,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
9,ses-11_date-20260116,11,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True



  MY_05_R


,session,ses_num,problem,signature,prob_changed,is_revisit
0,ses-1_date-20260111,1,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",True,False
1,ses-2_date-20260111,2,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
2,ses-3_date-20260112,3,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
3,ses-4_date-20260112,4,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
4,ses-5_date-20260113,5,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
5,ses-6_date-20260113,6,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
6,ses-7_date-20260114,7,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
7,ses-8_date-20260114,8,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
8,ses-9_date-20260115,9,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True
9,ses-10_date-20260115,10,1,"(((A, 1), (A, 3), (C, 3)), (B, 2))",False,True


## Problem transition summary
Shows only the sessions where the problem number changed, per subject.

In [4]:
for subj, grp in df.groupby("subject"):
    grp = grp.sort_values("ses_num").reset_index(drop=True)
    transitions = grp[grp["problem"] != grp["problem"].shift()]

    print(f"\n{subj} ({len(grp['problem'].unique())} problems):")
    for _, row in transitions.iterrows():
        print(f"  Problem {row['problem']:2d} @ {row['session']}: {row['signature']}")


MY_04_L (9 problems):
  Problem  1 @ ses-1_date-20260111: ((('A', 1), ('A', 3), ('C', 3)), ('B', 2))
  Problem  2 @ ses-25_date-20260123: ((('A', 1), ('B', 1), ('C', 1)), ('B', 3))
  Problem  3 @ ses-37_date-20260129: ((('A', 2), ('C', 1), ('C', 3)), ('B', 2))
  Problem  4 @ ses-44_date-20260201: ((('A', 2), ('B', 2), ('C', 2)), ('B', 1))
  Problem  5 @ ses-58_date-20260210: ((('A', 1), ('B', 3), ('C', 2)), ('B', 2))
  Problem  6 @ ses-67_date-20260215: ((('A', 1), ('A', 2), ('A', 3)), ('B', 2))
  Problem  7 @ ses-73_date-20260218: ((('A', 1), ('C', 1), ('C', 3)), ('B', 2))
  Problem  8 @ ses-85_date-20260227: ((('A', 1), ('A', 3), ('C', 2)), ('B', 2))
  Problem  9 @ ses-93_date-20260304: ((('A', 2), ('B', 1), ('C', 3)), ('B', 2))

MY_04_N (14 problems):
  Problem  1 @ ses-1_date-20260111: ((('A', 1), ('A', 3), ('C', 3)), ('B', 2))
  Problem  2 @ ses-25_date-20260123: ((('A', 2), ('B', 2), ('C', 2)), ('B', 3))
  Problem  3 @ ses-38_date-20260129: ((('A', 2), ('C', 1), ('C', 3)), ('B',

## Signature consistency check
Flags any problems where a subject has more than one signature (should never happen).

In [5]:
issues = []
for (subj, prob), grp in df.groupby(["subject", "problem"]):
    sigs = grp["signature"].dropna().unique()
    if len(sigs) > 1:
        issues.append({"subject": subj, "problem": prob, "signatures": list(sigs)})

if issues:
    print("ISSUES FOUND:")
    for issue in issues:
        print(f"  {issue['subject']} Problem {issue['problem']}: {issue['signatures']}")
else:
    print("All good — every subject has a single signature per problem.")

All good — every subject has a single signature per problem.


In [6]:
import json
for ses_key in ["ses-25_date-20260123", "ses-26_date-20260123"]:
    sess = subjects_data["MY_05_L"][ses_key]
    print(f"{ses_key}: initiation_tower={sess.get('initiation_tower')}")

ses-25_date-20260123: initiation_tower=B3
ses-26_date-20260123: initiation_tower=B3


In [7]:
import json
sess_df = subjects_data["MY_05_L"]["ses-25_date-20260123"]["df"]
if not isinstance(sess_df, list):
    sess_df = [sess_df]
for df in sess_df:
    mask = (df.get("type") == "variable") & (df.get("subtype") == "run_start")
    if mask.any():
        run_start = json.loads(df.loc[mask, "content"].iloc[0])
        print({k: v for k, v in run_start.items() if "init" in k.lower()})

{'init_cue_dur': 500.0}
